# Delete Exposures and Nights

Interactively delete exposures and nights of observations after inspect_nights.

Author: Qiushi Chris Tian

Created: 2026-09-28

<!-- Updated: -->

## Preamble

In [2]:
# %matplotlib ipympl

In [9]:
# from dep import plt, va
import numpy as np
from pathlib import Path
from astropy.table import Table
from newport import get_input_path, TARGET_GAIA_DR3

In [5]:
input_path = get_input_path(TARGET)
print(f'Reading original table from: {input_path}\n')

t = Table.read(input_path)
print(f'Original length: {len(t)}\n')

Reading original table from: data/tables/opt_comp_stars/TOI-1410/phot_w_err_TOI-1410_manual_del.fits

Original length: 2040



## Setting/Config

In [29]:
TARGET = 'TOI-1410'

WHOLE_NIGHTS = []
BAND_NIGHTS = {}
JDS = []
DATETIMES = ['2023-09-07T04:10:29.989', '2023-09-07T04:10:57.460', '2023-09-07T04:11:24.750']

print(
    f'{len(WHOLE_NIGHTS)} whole night(s), '
    f'{len(BAND_NIGHTS)} band night(s), '
    f'{len(JDS)} individual exposure(s) from JD, and '
    f'{len(DATETIMES)} individual exposure(s) from datetime'
)

PREVIEW_COLS = ['band', 'night', 'time', 'jd', 'airmass', 'exptime', TARGET_GAIA_DR3[TARGET]]#, 'skytemp']

0 whole night(s), 0 band night(s), 0 individual exposure(s) from JD, and 3 individual exposure(s) from datetime


## Inspect Rows in Table

In [12]:
pt = t[PREVIEW_COLS][(t['night'] == 20230906) & (t['band'] == 'R')]
pt.sort('jd')
pt.pprint_all()

band  night             time                  jd            airmass    exptime 1958584565350234752
---- -------- ----------------------- ------------------ ------------- ------- -------------------
   R 20230906 2023-09-07T03:13:30.539   2460194.63438125 1.01533216175    15.0   513588.9692178515
   R 20230906 2023-09-07T03:13:57.950 2460194.6346984953 1.01507269278    15.0   390337.3255305022
   R 20230906 2023-09-07T03:14:25.338  2460194.635015509 1.01481386429    15.0   291928.8032452726
   R 20230906 2023-09-07T03:14:52.700  2460194.635332176 1.01455812637    15.0   447509.8062305205
   R 20230906 2023-09-07T03:15:20.050  2460194.635648727 1.01430432271    15.0  516019.72859446786
   R 20230906 2023-09-07T03:32:30.050 2460194.6475700233 1.00643851178    15.0   301732.1796690084
   R 20230906 2023-09-07T03:32:57.380  2460194.647886343 1.00627224268    15.0  470354.75483121607
   R 20230906 2023-09-07T03:33:24.890 2460194.6482047453 1.00610874024    15.0   307448.2740583123
   R 20230

In [13]:
bt = Table.read('data/tables/opt_comp_stars/TOI-1410/20260930T21_man_del_c8/unbin_results_R.fits')

In [22]:
pvc = PREVIEW_COLS[1:]

bts = bt[bt['night'] == 20230906]
bts.sort('jd')
bts.pprint_all()

 night           jd            airmass    exptime        flux                error        
-------- ------------------ ------------- ------- ------------------ ---------------------
20230906   2460194.63438125 1.01533216175    15.0 1.0047378864581007 0.0016730256527813481
20230906 2460194.6346984953 1.01507269278    15.0 1.0067917577417496 0.0014732389705614623
20230906  2460194.635015509 1.01481386429    15.0 1.0117498514184098 0.0017206450135381973
20230906  2460194.635332176 1.01455812637    15.0 1.0003385550673651 0.0014093267516774029
20230906  2460194.635648727 1.01430432271    15.0 1.0108762051759514 0.0016970356503291453
20230906 2460194.6475700233 1.00643851178    15.0 1.0141096572032846 0.0016947145639091767
20230906  2460194.647886343 1.00627224268    15.0 1.0040415242560672 0.0014090024593752183
20230906 2460194.6482047453 1.00610874024    15.0 1.0077731910063847 0.0016794107188191965
20230906  2460194.648523611 1.00594672962    15.0  1.010025030337927  0.001721327877927962

## Building Mask

In [30]:
del_mask = np.zeros(len(t), dtype=bool)

for wn in WHOLE_NIGHTS:
    selected = t['night'] == wn
    print(f'{np.sum(selected)} exposure(s) on {wn} ...')
    del_mask |= selected

for night, bands in BAND_NIGHTS.items():
    for band in bands:
        selected = (t['band'] == band) & (t['night'] == night)
        print(f'{np.sum(selected)} exposure(s) in {band} band on {night} ...')
        del_mask |= selected

for jd in JDS:
    selected = t['jd'] == jd
    print(f'1 exposure on {t[selected]["night"][0]} ...')
    del_mask |= selected

for dt in DATETIMES:
    selected = t['time'] == dt
    print(f'1 exposure on {t[selected]["night"][0]} ...')
    del_mask |= selected

print('Done!')

1 exposure on 20230906 ...
1 exposure on 20230906 ...
1 exposure on 20230906 ...
Done!


## Preview Deletion

In [31]:
dt = t[del_mask]

print(f'Number of exposure(s): {len(dt)}')
print(f'Unique night(s): {len(np.unique(dt['night']))}\n')

dt[PREVIEW_COLS].pprint_all()

print(f'\nAfter deletion, length will become {len(t) - len(dt)}')

Number of exposure(s): 3
Unique night(s): 1

band  night             time                  jd            airmass    exptime 1958584565350234752
---- -------- ----------------------- ------------------ ------------- ------- -------------------
   R 20230906 2023-09-07T04:11:24.750  2460194.674592014  1.0002998395    15.0   512747.3486711385
   R 20230906 2023-09-07T04:10:29.989 2460194.6739582177 1.00026084513    15.0   510763.6237490808
   R 20230906 2023-09-07T04:10:57.460 2460194.6742761573 1.00027932621    15.0  376096.78597381467

After deletion, length will become 2037


## Executing Deletion

In [32]:
new_t = t[~del_mask]

print(f'New length: {len(new_t)}')

New length: 2037


## Save Table

In [33]:
save_path = f'data/tables/opt_comp_stars/{TARGET}/{input_path.stem}_manual_del{input_path.suffix}'
print(f'Modified table will be saved to: {save_path}\n')

Modified table will be saved to: data/tables/opt_comp_stars/TOI-1410/phot_w_err_TOI-1410_manual_del_manual_del.fits



In [34]:
if not Path(save_path).exists() or input(f'{save_path} exists. Overwrite? (y/*)').casefold().startswith('y'):
    new_t.write(save_path, overwrite=True)
    print(f'Saved to {save_path}')
else:
    print(f'Saving aborted')

Saved to data/tables/opt_comp_stars/TOI-1410/phot_w_err_TOI-1410_manual_del_manual_del.fits
